## 1. Simulasi form inputan dan analisa UPOS

In [ ]:
import stanza
import ipywidgets as widgets
from IPython.display import display, HTML

# 1. Inisialisasi Stanza untuk Bahasa Indonesia
print("Memuat model Stanza Bahasa Indonesia (mohon tunggu)...")
stanza.download('id', processors='tokenize,pos') # Menghapus quiet=True
nlp = stanza.Pipeline(lang='id', processors='tokenize,pos', verbose=False)
print("Model Stanza siap digunakan!\n")

# 2. Skema Warna untuk Tag UPOS agar visualisasi menarik
COLOR_MAP = {
    'NOUN': '#e3f2fd',    # Biru muda
    'VERB': '#e8f5e9',    # Hijau muda
    'ADJ': '#fff3e0',     # Oranye muda
    'PROPN': '#f3e5f5',   # Ungu muda
    'ADV': '#fffde7',     # Kuning muda
    'PRON': '#fce4ec',    # Pink muda
    'ADP': '#efebe9',     # Cokelat muda
    'AUX': '#e0f7fa',     # Toska muda
    'PUNCT': '#eceff1',   # Abu-abu muda
}

# 3. Membuat Komponen UI (Widgets)
text_input = widgets.Textarea(
    value='Budi sedang membaca buku baru di perpustakaan.',
    placeholder='Ketik kalimat bahasa Indonesia di sini...',
    description='Kalimat:',
    layout=widgets.Layout(width='80%', height='80px')
)

predict_button = widgets.Button(
    description='Analisis UPOS',
    button_style='primary',
    tooltip='Klik untuk menganalisis teks',
    icon='search'
)

output_area = widgets.Output()

# 4. Fungsi untuk Memproses Teks dan Merender HTML
def analyze_text(b):
    with output_area:
        output_area.clear_output() # Bersihkan hasil sebelumnya
        
        teks = text_input.value.strip()
        if not teks:
            print("Silakan masukkan teks terlebih dahulu.")
            return
        
        # Proses menggunakan Stanza
        doc = nlp(teks)
        
        # Mulai membangun struktur HTML untuk kartu kata
        html_cards = "<div style='display: flex; flex-wrap: wrap; gap: 10px; margin-top: 15px;'>"
        
        table_rows = ""
        
        for sentence in doc.sentences:
            for word in sentence.words:
                bg_color = COLOR_MAP.get(word.upos, '#ffffff') # Default putih jika tag tidak terdaftar
                
                # Render HTML untuk Visualisasi Kartu
                html_cards += f"""
                <div style='background-color: {bg_color}; border: 1px solid #ccc; border-radius: 6px; padding: 8px 12px; text-align: center; min-width: 70px; box-shadow: 1px 1px 3px rgba(0,0,0,0.1);'>
                    <strong style='font-size: 16px; color: #333;'>{word.text}</strong>
                    <div style='font-size: 11px; margin-top: 4px; font-weight: bold; color: #555; background: rgba(255,255,255,0.6); padding: 2px 4px; border-radius: 4px;'>{word.upos}</div>
                </div>
                """
                
                # Render Baris Tabel untuk Detail Fitur
                feats = word.feats if word.feats else "-"
                table_rows += f"<tr><td style='padding:8px; border-bottom:1px solid #ddd;'>{word.text}</td><td style='padding:8px; border-bottom:1px solid #ddd;'><b>{word.upos}</b></td><td style='padding:8px; border-bottom:1px solid #ddd; font-family:monospace; font-size:12px;'>{feats}</td></tr>"
                
        html_cards += "</div>"
        
        # Render Tabel Lengkap
        html_table = f"""
        <h4 style='margin-top: 25px; margin-bottom: 10px;'>Tabel Detail Analisis</h4>
        <table style='width: 80%; border-collapse: collapse; text-align: left; background: white; border: 1px solid #ddd;'>
            <thead>
                <tr style='background-color: #f5f5f5;'>
                    <th style='padding:10px; border-bottom: 2px solid #ddd;'>Kata</th>
                    <th style='padding:10px; border-bottom: 2px solid #ddd;'>Tag UPOS</th>
                    <th style='padding:10px; border-bottom: 2px solid #ddd;'>Fitur Morfologi (Features)</th>
                </tr>
            </thead>
            <tbody>
                {table_rows}
            </tbody>
        </table>
        """
        
        # Tampilkan hasil ke Output Widget
        display(HTML("<h4 style='margin-bottom: 5px;'>Hasil Visualisasi Tag:</h4>"))
        display(HTML(html_cards))
        display(HTML(html_table))

# 5. Menyambungkan Tombol Klik dengan Fungsi Analisis
predict_button.on_click(analyze_text)

# 6. Menampilkan UI di Notebook
ui_layout = widgets.VBox([
    text_input, 
    widgets.Box([predict_button], layout=widgets.Layout(margin='10px 0px 10px 0px')),
    output_area
])
display(ui_layout)

## Text Classification dengan Scikit-learn dan NLTK dalam prediksi kelulusan siswa

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re

# Pastikan NLTK data sudah diunduh. Jika belum, uncomment baris ini:
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# Dapatkan daftar stop words bahasa Indonesia (asumsi sudah dimuat di sel sebelumnya)
# Jika belum, definisikan di sini:
# stop_words_id = set(stopwords.words('indonesian'))

# Definisi ulang preprocess_text jika belum ada di scope saat ini
# (Ini penting untuk memastikan fungsi tersedia)
if 'preprocess_text' not in globals():
    stop_words_id = set(stopwords.words('indonesian'))
    def preprocess_text(text):
        text = text.lower()
        text = re.sub(r'[^a-z\s]', '', text)
        tokens = word_tokenize(text)
        filtered_tokens = [word for word in tokens if word not in stop_words_id and len(word) > 1]
        return ' '.join(filtered_tokens)


print("\n--- Contoh Klasifikasi Teks: Prediksi Kelulusan Siswa ---")

# --- 1. Persiapan Data Simulasi Kelulusan Siswa ---
data_kelulusan = {
    'deskripsi_performa': [
        "Siswa ini sangat rajin dan memiliki nilai yang sangat bagus.", # lulus
        "Tidak pernah masuk kelas, tugas tidak dikumpul, sering bolos.", # tidak_lulus
        "Rata-rata di semua mata pelajaran, cukup aktif di kelas.", # lulus
        "Sangat malas, mencontek saat ujian, sering tidak mengerjakan PR.", # tidak_lulus
        "Menunjukkan potensi besar, namun kurang konsisten dalam belajar.", # lulus
        "Selalu bersemangat, mengerjakan tugas dengan sempurna.", # lulus
        "Kurang motivasi, nilai sering kurang, jarang berpartisipasi.", # tidak_lulus
        "Memiliki nilai terbaik, mandiri, dan sangat aktif di semua kegiatan.", # lulus
        "Butuh banyak perbaikan, nilai signifikan rendah, cenderung pasif dan kurang peduli.", # tidak_lulus
        "Sering berusaha keras, namun kadang kesulitan dan mudah menyerah.", # lulus (dengan bimbingan)
        # --- Penambahan Data Baru untuk Meningkatkan Kualitas Model ---
        "Selalu proaktif dalam tugas kelompok dan presentasi, nilainya konsisten.", # lulus
        "Sering menunda-nunda pekerjaan, hasil tugas tidak maksimal, dan kurang fokus.", # tidak_lulus
        "Nilai rata-rata di atas standar, menunjukkan pemahaman materi yang mendalam.", # lulus
        "Tidak fokus di kelas, sering mengganggu teman lain, dan tidak mendengarkan guru.", # tidak_lulus
        "Siswa dengan etos kerja tinggi, sangat bertanggung jawab terhadap kewajibannya.", # lulus
        "Kesulitan memahami konsep dasar, tidak ada usaha perbaikan, dan mudah menyerah.", # tidak_lulus
        "Mampu bekerja sama dengan baik dan memberikan kontribusi positif dalam diskusi.", # lulus
        "Absensi sangat buruk, tidak mengikuti ujian penting, dan tidak ada kabar.", # tidak_lulus
        "Memiliki inisiatif tinggi untuk belajar hal baru di luar kurikulum dan berprestasi.", # lulus
        "Bersikap apatis terhadap pelajaran, tidak menunjukkan minat sama sekali, dan sering tidur di kelas.", # tidak_lulus
        "Plagiat dalam beberapa tugas, melanggar etika akademik, dan tidak menyesali perbuatannya.", # tidak_lulus
        "Tidak pernah berpartisipasi, cenderung diam dan pasif, serta takut untuk bertanya.", # tidak_lulus
        "Hasil ujian selalu di bawah rata-rata, tidak ada peningkatan signifikan, dan tidak belajar.", # tidak_lulus
        "Sering mencari alasan untuk tidak mengerjakan tugas, tidak bertanggung jawab, dan menyalahkan orang lain.", # tidak_lulus
        "Siswa yang selalu ingin tahu, selalu bertanya, dan memiliki semangat belajar yang tinggi.", # lulus
        "Cukup baik, tetapi perlu dorongan ekstra untuk mencapai potensi penuhnya.", # lulus
        "Sering membuat keributan di kelas dan tidak menghargai proses belajar.", # tidak_lulus
        "Sangat fokus dan selalu menyelesaikan tugas sebelum tenggat waktu.", # lulus
        "Tidak memahami instruksi, sering mengulang kesalahan yang sama, dan lambat dalam belajar.", # tidak_lulus
        "Memiliki kemampuan analisis yang baik dan selalu memberikan ide-ide segar.", # lulus
    ],
    'graduation_status': [
        'lulus',
        'tidak_lulus',
        'lulus',
        'tidak_lulus',
        'lulus',
        'lulus',
        'tidak_lulus',
        'lulus',
        'tidak_lulus',
        'lulus',
        # --- Status untuk Penambahan Data Baru ---
        'lulus',
        'tidak_lulus',
        'lulus',
        'tidak_lulus',
        'lulus',
        'tidak_lulus',
        'lulus',
        'tidak_lulus',
        'lulus',
        'tidak_lulus',
        'tidak_lulus',
        'tidak_lulus',
        'tidak_lulus',
        'tidak_lulus',
        'lulus',
        'lulus',
        'tidak_lulus',
        'lulus',
        'tidak_lulus',
        'lulus'
    ]
}
df_kelulusan = pd.DataFrame(data_kelulusan)
print("\nDataset untuk Prediksi Kelulusan Siswa:")
display(df_kelulusan)

# --- 2. Pra-pemrosesan Teks ---
df_kelulusan['cleaned_performance'] = df_kelulusan['deskripsi_performa'].apply(preprocess_text)
print("\nDataset setelah pra-pemrosesan:")
display(df_kelulusan)

# --- 3. Ekstraksi Fitur (TF-IDF) ---
X_kelulusan = df_kelulusan['cleaned_performance']
y_kelulusan = df_kelulusan['graduation_status']

X_train_kelulusan, X_test_kelulusan, y_train_kelulusan, y_test_kelulusan = train_test_split(
    X_kelulusan, y_kelulusan, test_size=0.3, random_state=42, stratify=y_kelulusan
)

print(f"\nJumlah data training kelulusan: {len(X_train_kelulusan)}")
print(f"Jumlah data testing kelulusan: {len(X_test_kelulusan)}")

tfidf_vectorizer_kelulusan = TfidfVectorizer()
X_train_tfidf_kelulusan = tfidf_vectorizer_kelulusan.fit_transform(X_train_kelulusan)
X_test_tfidf_kelulusan = tfidf_vectorizer_kelulusan.transform(X_test_kelulusan)

print("Bentuk X_train_tfidf_kelulusan:", X_train_tfidf_kelulusan.shape)
print("Bentuk X_test_tfidf_kelulusan:", X_test_tfidf_kelulusan.shape)

# --- 4. Pelatihan Model (SVM) ---
svm_model_kelulusan = SVC(kernel='linear', random_state=42)
svm_model_kelulusan.fit(X_train_tfidf_kelulusan, y_train_kelulusan)
print("\nModel SVM untuk kelulusan siswa telah berhasil dilatih.")

# --- 5. Evaluasi Model ---
y_pred_kelulusan = svm_model_kelulusan.predict(X_test_tfidf_kelulusan)
accuracy_kelulusan = accuracy_score(y_test_kelulusan, y_pred_kelulusan)
print(f"\nAkurasi Model Kelulusan: {accuracy_kelulusan:.2f}")
report_kelulusan = classification_report(y_test_kelulusan, y_pred_kelulusan, zero_division=0)
print("\nClassification Report Kelulusan:\n", report_kelulusan)

# --- 6. Uji Model dengan Kalimat Input Baru ---
new_student_performance = [
    "Siswa sering absen dan nilai ujiannya sangat rendah.",
    "Aktif dalam diskusi kelas, mengerjakan tugas tepat waktu.",
    "Terlihat bingung dengan materi, perlu bimbingan ekstra."
]

df_new_performance = pd.DataFrame({'deskripsi_performa': new_student_performance})
df_new_performance['cleaned_performance'] = df_new_performance['deskripsi_performa'].apply(preprocess_text)

X_new_tfidf_kelulusan = tfidf_vectorizer_kelulusan.transform(df_new_performance['cleaned_performance'])
new_predictions_kelulusan = svm_model_kelulusan.predict(X_new_tfidf_kelulusan)

df_new_performance['predicted_status'] = new_predictions_kelulusan

print("\nPrediksi Kelulusan untuk Deskripsi Siswa Baru:")
display(df_new_performance)



--- Contoh Klasifikasi Teks: Prediksi Kelulusan Siswa ---

Dataset untuk Prediksi Kelulusan Siswa:


,deskripsi_performa,graduation_status
0,Siswa ini sangat rajin dan memiliki nilai yang...,lulus
1,"Tidak pernah masuk kelas, tugas tidak dikumpul...",tidak_lulus
2,"Rata-rata di semua mata pelajaran, cukup aktif...",lulus
3,"Sangat malas, mencontek saat ujian, sering tid...",tidak_lulus
4,"Menunjukkan potensi besar, namun kurang konsis...",lulus
5,"Selalu bersemangat, mengerjakan tugas dengan s...",lulus
6,"Kurang motivasi, nilai sering kurang, jarang b...",tidak_lulus
7,"Memiliki nilai terbaik, mandiri, dan sangat ak...",lulus
8,"Butuh banyak perbaikan, nilai signifikan renda...",tidak_lulus
9,"Sering berusaha keras, namun kadang kesulitan ...",lulus



Dataset setelah pra-pemrosesan:


,deskripsi_performa,graduation_status,cleaned_performance
0,Siswa ini sangat rajin dan memiliki nilai yang...,lulus,siswa rajin memiliki nilai bagus
1,"Tidak pernah masuk kelas, tugas tidak dikumpul...",tidak_lulus,masuk kelas tugas dikumpul bolos
2,"Rata-rata di semua mata pelajaran, cukup aktif...",lulus,ratarata mata pelajaran aktif kelas
3,"Sangat malas, mencontek saat ujian, sering tid...",tidak_lulus,malas mencontek ujian pr
4,"Menunjukkan potensi besar, namun kurang konsis...",lulus,potensi konsisten belajar
5,"Selalu bersemangat, mengerjakan tugas dengan s...",lulus,bersemangat tugas sempurna
6,"Kurang motivasi, nilai sering kurang, jarang b...",tidak_lulus,motivasi nilai jarang berpartisipasi
7,"Memiliki nilai terbaik, mandiri, dan sangat ak...",lulus,memiliki nilai terbaik mandiri aktif kegiatan
8,"Butuh banyak perbaikan, nilai signifikan renda...",tidak_lulus,butuh perbaikan nilai signifikan rendah cender...
9,"Sering berusaha keras, namun kadang kesulitan ...",lulus,berusaha keras kadang kesulitan mudah menyerah



Jumlah data training kelulusan: 21
Jumlah data testing kelulusan: 9
Bentuk X_train_tfidf_kelulusan: (21, 83)
Bentuk X_test_tfidf_kelulusan: (9, 83)

Model SVM untuk kelulusan siswa telah berhasil dilatih.

Akurasi Model Kelulusan: 0.78

Classification Report Kelulusan:
               precision    recall  f1-score   support

       lulus       0.80      0.80      0.80         5
 tidak_lulus       0.75      0.75      0.75         4

    accuracy                           0.78         9
   macro avg       0.78      0.78      0.78         9
weighted avg       0.78      0.78      0.78         9


Prediksi Kelulusan untuk Deskripsi Siswa Baru:


,deskripsi_performa,cleaned_performance,predicted_status
0,Siswa sering absen dan nilai ujiannya sangat r...,siswa absen nilai ujiannya rendah,lulus
1,"Aktif dalam diskusi kelas, mengerjakan tugas t...",aktif diskusi kelas tugas,lulus
2,"Terlihat bingung dengan materi, perlu bimbinga...",bingung materi bimbingan ekstra,tidak_lulus


### Simpan model prediksi kelulusan siswa

In [ ]:
import joblib

# Tentukan nama file untuk menyimpan model kelulusan dan vectorizer
model_kelulusan_filename = 'svm_kelulusan_model.joblib'
vectorizer_kelulusan_filename = 'tfidf_kelulusan_vectorizer.joblib'

# Simpan model SVM untuk prediksi kelulusan
joblib.dump(svm_model_kelulusan, model_kelulusan_filename)
print(f"Model SVM Kelulusan telah disimpan ke '{model_kelulusan_filename}'")

# Simpan TF-IDF Vectorizer untuk prediksi kelulusan
joblib.dump(tfidf_vectorizer_kelulusan, vectorizer_kelulusan_filename)
print(f"TF-IDF Vectorizer Kelulusan telah disimpan ke '{vectorizer_kelulusan_filename}'")

### form inputan pengujian prediksi kelulusan siswa

In [ ]:
import joblib
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd

# Memuat model dan vectorizer kelulusan siswa yang telah disimpan
# Pastikan nama file sesuai dengan yang disimpan di sel JoUOK2RF9oIZ
model_kelulusan_filename = 'svm_kelulusan_model.joblib'
vectorizer_kelulusan_filename = 'tfidf_kelulusan_vectorizer.joblib'

try:
    loaded_svm_kelulusan_model = joblib.load(model_kelulusan_filename)
    loaded_tfidf_kelulusan_vectorizer = joblib.load(vectorizer_kelulusan_filename)
    print("Model SVM Kelulusan dan TF-IDF Vectorizer berhasil dimuat.")
except FileNotFoundError:
    print(f"Error: Model atau Vectorizer kelulusan siswa tidak ditemukan. Pastikan '{model_kelulusan_filename}' dan '{vectorizer_kelulusan_filename}' sudah disimpan.")
    loaded_svm_kelulusan_model = None
    loaded_tfidf_kelulusan_vectorizer = None


# Buat widget input
performance_input = widgets.Textarea(
    value='',
    placeholder='Masukkan deskripsi performa siswa di sini (misal: rajin belajar dan nilai bagus)',
    description='Performa Siswa:',
    disabled=False,
    layout=widgets.Layout(width='500px', height='100px')
)

predict_button_kelulusan = widgets.Button(
    description='Prediksi Kelulusan',
    disabled=False,
    button_style='info',
    tooltip='Klik untuk memprediksi status kelulusan',
    icon='search'
)

output_area_kelulusan = widgets.Output()

def on_predict_button_kelulusan_clicked(b):
    with output_area_kelulusan:
        clear_output()
        if loaded_svm_kelulusan_model is None or loaded_tfidf_kelulusan_vectorizer is None:
            print("Model kelulusan belum dimuat. Tidak dapat melakukan prediksi.")
            return

        user_performance = performance_input.value
        if not user_performance.strip():
            print("Mohon masukkan deskripsi performa siswa.")
            return

        print(f"Deskripsi performa siswa: '{user_performance}'")

        # Pra-pemrosesan teks
        preprocessed_performance = preprocess_text(user_performance)
        print(f"Teks setelah pra-pemrosesan: '{preprocessed_performance}'")

        if not preprocessed_performance:
            print("Tidak ada kata kunci yang tersisa setelah pra-pemrosesan. Tidak dapat memprediksi.")
            return

        # Mengubah teks yang telah diproses menjadi fitur TF-IDF
        X_user_performance_tfidf = loaded_tfidf_kelulusan_vectorizer.transform([preprocessed_performance])

        # Melakukan prediksi
        predicted_status = loaded_svm_kelulusan_model.predict(X_user_performance_tfidf)

        print(f"\nStatus kelulusan yang diprediksi: '{predicted_status[0]}'")

# Menghubungkan event klik tombol ke fungsi
predict_button_kelulusan.on_click(on_predict_button_kelulusan_clicked)

# Menampilkan widget
display(performance_input, predict_button_kelulusan, output_area_kelulusan)

## Sentiment Analysis & Text Classification

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re
import joblib
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Pastikan NLTK data sudah diunduh ---
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# --- Definisi fungsi pra-pemrosesan teks ---
stop_words_id = set(stopwords.words('indonesian'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text) # Hanya menyisakan huruf kecil dan spasi
    tokens = word_tokenize(text)
    filtered_tokens = [word for word in tokens if word not in stop_words_id and len(word) > 1]
    return ' '.join(filtered_tokens)

# --- Fungsi untuk menghasilkan Ajuran ---
def generate_ajuran(predicted_status, original_description):
    if predicted_status == 'lulus':
        return "Pertahankan kinerja baik ini dan terus kembangkan potensi! Jangan mudah puas, teruslah berinovasi dan jadi inspirasi bagi sesama."
    else:
        ajuran_detail = ""
        # Perbaikan: Menggunakan kata kunci yang lebih generik atau mencari pola
        if any(keyword in original_description.lower() for keyword in ['absen', 'bolos', 'tidak masuk']):
            ajuran_detail += "Perlu meningkatkan kehadiran di kelas."
        if any(keyword in original_description.lower() for keyword in ['nilai rendah', 'kurang motivasi', 'sulit', 'tidak paham', 'bingung']):
            if ajuran_detail: ajuran_detail += " Selain itu, "
            ajuran_detail += "fokus pada peningkatan nilai dan mencari motivasi belajar."
        if any(keyword in original_description.lower() for keyword in ['pasif', 'kurang peduli', 'diam', 'tidak berpartisipasi']):
            if ajuran_detail: ajuran_detail += " Serta, "
            ajuran_detail += "lebih aktif berpartisipasi dan menunjukkan kepedulian dalam pembelajaran."
        if any(keyword in original_description.lower() for keyword in ['tugas tidak dikumpul', 'tidak mengerjakan pr', 'menunda-nunda']):
            if ajuran_detail: ajuran_detail += " Dan, "
            ajuran_detail += "disiplin dalam mengerjakan dan mengumpulkan tugas."
        if 'mencontek' in original_description.lower() or 'plagiat' in original_description.lower():
            if ajuran_detail: ajuran_detail += " Dan, "
            ajuran_detail += "menjaga integritas akademik dengan tidak mencontek atau plagiat."

        if not ajuran_detail:
            ajuran_detail = "Perlu identifikasi area kelemahan dan segera mencari bimbingan untuk meningkatkan performa."

        return f"Perlu perhatian serius! {ajuran_detail} Ingat, kegagalan adalah guru terbaik. Bangkit dan belajar dari kesalahan!"


print("---" * 15)
print("Contoh Klasifikasi Sentimen Kelulusan Siswa (Naive Bayes)")
print("---" * 15)

# --- 1. Persiapan Data Simulasi Kelulusan Siswa ---
data_kelulusan_sa = {
    'deskripsi_performa': [
        "Siswa ini sangat rajin dan memiliki nilai yang sangat bagus.", # lulus
        "Tidak pernah masuk kelas, tugas tidak dikumpul, sering bolos.", # tidak_lulus
        "Rata-rata di semua mata pelajaran, cukup aktif di kelas.", # lulus
        "Sangat malas, mencontek saat ujian, sering tidak mengerjakan PR.", # tidak_lulus
        "Menunjukkan potensi besar, namun kurang konsisten dalam belajar.", # lulus
        "Selalu bersemangat, mengerjakan tugas dengan sempurna.", # lulus
        "Kurang motivasi, nilai sering kurang, jarang berpartisipasi.", # tidak_lulus
        "Memiliki nilai terbaik, mandiri, dan sangat aktif di semua kegiatan.", # lulus
        "Butuh banyak perbaikan, nilai signifikan rendah, cenderung pasif dan kurang peduli.", # tidak_lulus
        "Sering berusaha keras, namun kadang kesulitan dan mudah menyerah.", # lulus (dengan bimbingan)
        # --- Penambahan Data Baru untuk Meningkatkan Kualitas Model ---
        "Selalu proaktif dalam tugas kelompok dan presentasi, nilainya konsisten.", # lulus
        "Sering menunda-nunda pekerjaan, hasil tugas tidak maksimal, dan kurang fokus.", # tidak_lulus
        "Nilai rata-rata di atas standar, menunjukkan pemahaman materi yang mendalam.", # lulus
        "Tidak fokus di kelas, sering mengganggu teman lain, dan tidak mendengarkan guru.", # tidak_lulus
        "Siswa dengan etos kerja tinggi, sangat bertanggung jawab terhadap kewajibannya.", # lulus
        "Kesulitan memahami konsep dasar, tidak ada usaha perbaikan, dan mudah menyerah.", # tidak_lulus
        "Mampu bekerja sama dengan baik dan memberikan kontribusi positif dalam diskusi.", # lulus
        "Absensi sangat buruk, tidak mengikuti ujian penting, dan tidak ada kabar.", # tidak_lulus
        "Memiliki inisiatif tinggi untuk belajar hal baru di luar kurikulum dan berprestasi.", # lulus
        "Bersikap apatis terhadap pelajaran, tidak menunjukkan minat sama sekali, dan sering tidur di kelas.", # tidak_lulus
        "Plagiat dalam beberapa tugas, melanggar etika akademik, dan tidak menyesali perbuatannya.", # tidak_lulus
        "Tidak pernah berpartisipasi, cenderung diam dan pasif, serta takut untuk bertanya.", # tidak_lulus
        "Hasil ujian selalu di bawah rata-rata, tidak ada peningkatan signifikan, dan tidak belajar.", # tidak_lulus
        "Sering mencari alasan untuk tidak mengerjakan tugas, tidak bertanggung jawab, dan menyalahkan orang lain.", # tidak_lulus
        "Siswa yang selalu ingin tahu, selalu bertanya, dan memiliki semangat belajar yang tinggi.", # lulus
        "Cukup baik, tetapi perlu dorongan ekstra untuk mencapai potensi penuhnya.", # lulus
        "Sering membuat keributan di kelas dan tidak menghargai proses belajar.", # tidak_lulus
        "Sangat fokus dan selalu menyelesaikan tugas sebelum tenggat waktu.", # lulus
        "Tidak memahami instruksi, sering mengulang kesalahan yang sama, dan lambat dalam belajar.", # tidak_lulus
        "Memiliki kemampuan analisis yang baik dan selalu memberikan ide-ide segar.", # lulus
    ],
    'status_kelulusan': [
        'lulus',
        'tidak_lulus',
        'lulus',
        'tidak_lulus',
        'lulus',
        'lulus',
        'tidak_lulus',
        'lulus',
        'tidak_lulus',
        'lulus',
        # --- Status untuk Penambahan Data Baru ---
        'lulus',
        'tidak_lulus',
        'lulus',
        'tidak_lulus',
        'lulus',
        'tidak_lulus',
        'lulus',
        'tidak_lulus',
        'lulus',
        'tidak_lulus',
        'tidak_lulus',
        'tidak_lulus',
        'tidak_lulus',
        'tidak_lulus',
        'lulus',
        'lulus',
        'tidak_lulus',
        'lulus',
        'tidak_lulus',
        'lulus'
    ]
}

df_kelulusan_sa = pd.DataFrame(data_kelulusan_sa)
print("\nDataset untuk Klasifikasi Sentimen Kelulusan Siswa:")
display(df_kelulusan_sa.head())

# --- 2. Pra-pemrosesan Teks ---
df_kelulusan_sa['cleaned_performa'] = df_kelulusan_sa['deskripsi_performa'].apply(preprocess_text)
print("\nDataset setelah pra-pemrosesan:")
display(df_kelulusan_sa.head())

# --- 3. Ekstraksi Fitur (TF-IDF) ---
X_kelulusan_sa = df_kelulusan_sa['cleaned_performa']
y_kelulusan_sa = df_kelulusan_sa['status_kelulusan']

X_train_sa, X_test_sa, y_train_sa, y_test_sa = train_test_split(
    X_kelulusan_sa, y_kelulusan_sa, test_size=0.25, random_state=42, stratify=y_kelulusan_sa # Adjusted test_size and added stratify
)

print(f"\nJumlah data training kelulusan: {len(X_train_sa)}")
print(f"Jumlah data testing kelulusan: {len(X_test_sa)}")

tfidf_vectorizer_sa = TfidfVectorizer()
X_train_tfidf_sa = tfidf_vectorizer_sa.fit_transform(X_train_sa)
X_test_tfidf_sa = tfidf_vectorizer_sa.transform(X_test_sa)

print("Bentuk X_train_tfidf_sa:", X_train_tfidf_sa.shape)
print("Bentuk X_test_tfidf_sa:", X_test_tfidf_sa.shape)

# --- 4. Pelatihan Model (Multinomial Naive Bayes) ---
mnb_model_sa = MultinomialNB()
mnb_model_sa.fit(X_train_tfidf_sa, y_train_sa)
print("\nModel Multinomial Naive Bayes untuk kelulusan siswa telah berhasil dilatih.")

# --- 5. Evaluasi Model ---
y_pred_sa = mnb_model_sa.predict(X_test_tfidf_sa)
accuracy_sa = accuracy_score(y_test_sa, y_pred_sa)
print(f"\nAkurasi Model Kelulusan (Naive Bayes): {accuracy_sa:.2f}")
report_sa = classification_report(y_test_sa, y_pred_sa, zero_division=0)
print("\nClassification Report Kelulusan (Naive Bayes):\n", report_sa)

# --- Interactive Form for Prediction and Advice ---

performance_input_sa = widgets.Textarea(
    value='',
    placeholder='Masukkan deskripsi performa siswa di sini (misal: rajin belajar dan nilai bagus)',
    description='Performa Siswa:',
    disabled=False,
    layout=widgets.Layout(width='80%', height='100px')
)

predict_button_sa = widgets.Button(
    description='Prediksi & Beri Ajuran',
    disabled=False,
    button_style='success',
    tooltip='Klik untuk memprediksi status kelulusan dan mendapatkan ajuran',
    icon='lightbulb'
)

output_area_sa = widgets.Output()

def on_predict_button_sa_clicked(b):
    with output_area_sa:
        clear_output()
        # Ensure model and vectorizer are available (they are trained in this cell)
        user_performance_desc = performance_input_sa.value
        if not user_performance_desc.strip():
            print("Mohon masukkan deskripsi performa siswa.")
            return

        print(f"Deskripsi performa siswa: '{user_performance_desc}'")

        # Pra-pemrosesan teks
        preprocessed_performance_sa = preprocess_text(user_performance_desc)
        print(f"Teks setelah pra-pemrosesan: '{preprocessed_performance_sa}'")

        if not preprocessed_performance_sa:
            print("Tidak ada kata kunci yang tersisa setelah pra-pemrosesan. Tidak dapat memprediksi.")
            return

        # Mengubah teks yang telah diproses menjadi fitur TF-IDF
        X_user_performance_tfidf_sa = tfidf_vectorizer_sa.transform([preprocessed_performance_sa])

        # Melakukan prediksi
        predicted_status_sa = mnb_model_sa.predict(X_user_performance_tfidf_sa)

        # Menghasilkan ajuran
        ajuran_message = generate_ajuran(predicted_status_sa[0], user_performance_desc)

        print(f"\nStatus kelulusan yang diprediksi: '{predicted_status_sa[0].upper()}'")
        print(f"Ajuran: {ajuran_message}")

# Menghubungkan event klik tombol ke fungsi
predict_button_sa.on_click(on_predict_button_sa_clicked)

# Menampilkan widget
display(performance_input_sa, predict_button_sa, output_area_sa)
